# Equations Reference — SX Phoenicis Distance Project

Every key formula in the project, with a plain-language statement, the equation,
a worked numerical example for **CY Aquarii**, and runnable code. Companion to
the narrative reference PDF.

Run top to bottom. No internet required — all numbers are hard-coded from the
project so it works offline (e.g. on a flight).

In [1]:
import numpy as np

# CY Aqr reference values used throughout (from the project)
P_CYAQR   = 0.06103845      # fundamental period, days
V_MEAN    = 10.893          # intensity-mean apparent V (all-nights fold)
D_GAIA    = 412.7           # Gaia Bailer-Jones geometric distance, pc
PLX       = 2.385           # Gaia parallax, mas
A_V       = 0.10            # assumed extinction

# Cohen & Sarajedini (2012) coefficients
A_CS, A_ERR = -1.640, 0.110
B_CS, B_ERR = -3.389, 0.090
SIGMA_INTRINSIC = 0.20
print("reference values loaded")

reference values loaded


## 1. Period–luminosity relation

**Plain language:** the pulsation period sets the star's true (absolute)
brightness. Longer period → brighter → more negative absolute magnitude.

$$M_V = a + b\,\log_{10}(P), \quad a=-1.640,\; b=-3.389$$

For a fundamental-mode period in days.

In [2]:
logP = np.log10(P_CYAQR)
M_V  = A_CS + B_CS * logP
print(f"log10(P) = {logP:.4f}   (negative, since P < 1 day)")
print(f"M_V      = {A_CS} + ({B_CS})*({logP:.4f}) = {M_V:.3f}")
# note: negative slope x negative logP -> positive contribution
print(f"\nCY Aqr predicted absolute magnitude M_V = {M_V:.2f}")

log10(P) = -1.2144   (negative, since P < 1 day)
M_V      = -1.64 + (-3.389)*(-1.2144) = 2.476

CY Aqr predicted absolute magnitude M_V = 2.48


## 2. Absolute magnitude uncertainty — two kinds

**Plain language:** the calibration error is how well we know the mean line; the
intrinsic scatter is how far a *single* star sits from it. For one star's
distance you need both, added in quadrature.

$$\sigma_{cal}^2 = \sigma_a^2 + (\log P \cdot \sigma_b)^2, \qquad
\sigma_{tot} = \sqrt{\sigma_{cal}^2 + \sigma_{intrinsic}^2}$$

In [3]:
sig_cal = np.sqrt(A_ERR**2 + (logP*B_ERR)**2)
sig_tot = np.sqrt(sig_cal**2 + SIGMA_INTRINSIC**2)
print(f"calibration-only sigma_M = {sig_cal:.3f} mag")
print(f"with intrinsic scatter   = {sig_tot:.3f} mag  <- use this for one star")
print(f"\nintrinsic scatter ({SIGMA_INTRINSIC}) dominates -> the precision floor")

calibration-only sigma_M = 0.155 mag
with intrinsic scatter   = 0.253 mag  <- use this for one star

intrinsic scatter (0.2) dominates -> the precision floor


## 3. Distance modulus → distance

**Plain language:** compare how bright the star *looks* (apparent m) to how bright
it *is* (absolute M); the difference, corrected for dust, gives distance.

$$m - M = 5\log_{10}(d) - 5 + A_V \;\Rightarrow\;
d = 10^{\,(m - M - A_V + 5)/5}$$

Distance is exponential in the modulus, so
$dd/d\mu = d\ln(10)/5$ carries the error through.

In [4]:
M_V_err = sig_tot
mu0 = V_MEAN - M_V - A_V
d   = 10**((mu0 + 5)/5)
# propagate: modulus error from m, M, A_V in quadrature
m_err, A_err = 0.02, 0.05
sig_mu = np.sqrt(m_err**2 + M_V_err**2 + A_err**2)
d_err  = d * (np.log(10)/5) * sig_mu
print(f"corrected distance modulus = {mu0:.3f}")
print(f"distance = {d:.0f} +/- {d_err:.0f} pc   ({100*d_err/d:.0f}%)")
print(f"Gaia answer key = {D_GAIA:.0f} pc")
print(f"\ngap = {d-D_GAIA:+.0f} pc = {(d-D_GAIA)/d_err:+.2f} sigma  (within 1 sigma)")

corrected distance modulus = 8.317
distance = 461 +/- 55 pc   (12%)
Gaia answer key = 413 pc

gap = +48 pc = +0.88 sigma  (within 1 sigma)


## 4. Absolute magnitude from an *independent* distance

**Plain language:** to place a star on the PL plot without circular reasoning, get
its true M_V from its apparent magnitude and an independent (Gaia) distance —
never from the PL relation itself.

$$M_V = m - 5\log_{10}(d) + 5 - A_V$$

In [5]:
M_true = V_MEAN - 5*np.log10(D_GAIA) + 5 - A_V
print(f"CY Aqr true M_V (from Gaia distance) = {M_true:.3f}")
print(f"PL-predicted M_V                     = {M_V:.3f}")
print(f"offset (true - predicted) = {M_true - M_V:+.3f} mag")
print(f"\n-> CY Aqr is ~{M_true-M_V:.2f} mag FAINTER than predicted:")
print(f"   an intrinsically faint outlier, which is why its PL distance overshot.")

CY Aqr true M_V (from Gaia distance) = 2.715
PL-predicted M_V                     = 2.476
offset (true - predicted) = +0.239 mag

-> CY Aqr is ~0.24 mag FAINTER than predicted:
   an intrinsically faint outlier, which is why its PL distance overshot.


## 5. Inverse-square law (the foundation)

**Plain language:** brightness falls with the square of distance. This underlies
the whole distance modulus.

$$F = \frac{L}{4\pi d^2}, \qquad \frac{F_1}{F_2} = \left(\frac{d_2}{d_1}\right)^2$$

And Pogson's magnitude definition (a factor of 100 in flux = 5 magnitudes):

$$m_1 - m_2 = -2.5\log_{10}(F_1/F_2)$$

In [6]:
# demonstrate: a star at 2x the distance is 4x fainter = +1.505 mag
for factor in [2, 10, 100]:
    dm = -2.5*np.log10(1/factor**2)
    print(f"at {factor:3d}x distance: {factor**2:5d}x fainter = {dm:+.3f} mag")
print(f"\n(100x fainter = exactly +5.000 mag, by Pogson's definition)")

at   2x distance:     4x fainter = +1.505 mag
at  10x distance:   100x fainter = +5.000 mag
at 100x distance: 10000x fainter = +10.000 mag

(100x fainter = exactly +5.000 mag, by Pogson's definition)


## 6. Period–mean-density relation (the pulsation clock)

**Plain language:** the period is a readout of the star's mean density — denser
stars pulsate faster. This is *why* period encodes size, hence luminosity.

$$P\sqrt{\bar\rho} = Q \;\Rightarrow\; P \propto \bar\rho^{-1/2}$$

In [7]:
# CY Aqr: estimate mean density from period (Q ~ 0.033 d for fundamental mode)
# rho in solar units; this is illustrative of the scaling, not a precise value
Q = 0.033  # days, approximate pulsation constant, fundamental radial mode
rho_over_rhosun = (Q / P_CYAQR)**2
print(f"P = {P_CYAQR:.5f} d")
print(f"mean density ~ (Q/P)^2 = {rho_over_rhosun:.2f} x solar mean density")
print(f"\n(illustrative: shorter period -> higher density -> smaller, denser star)")

P = 0.06104 d
mean density ~ (Q/P)^2 = 0.29 x solar mean density

(illustrative: shorter period -> higher density -> smaller, denser star)


## 7. Luminosity of a glowing sphere (closing the loop)

**Plain language:** luminosity depends on radius and temperature. With temperature
nearly fixed by the instability strip, a bigger (longer-period) star is more
luminous — the physical basis of the PL relation.

$$L = 4\pi R^2 \sigma T^4$$

In [8]:
# illustrate: at FIXED temperature, luminosity scales as radius^2
sigma_sb = 5.67e-8  # W m^-2 K^-4
# two stars, same T, radii differing by 30%
T = 7500.0  # K, typical for these A/F pulsators
for R_solar in [1.5, 2.0]:
    R = R_solar * 6.957e8  # m
    L = 4*np.pi*R**2 * sigma_sb * T**4
    print(f"R = {R_solar} Rsun, T = {T:.0f} K  ->  L = {L/3.828e26:.1f} Lsun")
print(f"\n-> larger radius (from longer period) => higher luminosity, at fixed T")

R = 1.5 Rsun, T = 7500 K  ->  L = 6.4 Lsun
R = 2.0 Rsun, T = 7500 K  ->  L = 11.4 Lsun

-> larger radius (from longer period) => higher luminosity, at fixed T


## 8. Intensity mean (the correct average brightness)

**Plain language:** average in *flux* (linear), not magnitude (log), and weight
every *phase* equally. A naive magnitude-average is biased faint.

$$\langle V\rangle = -2.5\log_{10}\!\left(\frac1N\sum_i 10^{-0.4 V_i}\right)$$

In [9]:
# demo: a star sampled unevenly across a 0.7-mag swing
np.random.seed(0)
# simulate: more samples near minimum (fainter), as real sawtooth data would be
mags = np.concatenate([np.full(70, 11.1), np.full(30, 10.4)])  # phase-biased
naive = mags.mean()
flux  = 10**(-0.4*mags)
intensity = -2.5*np.log10(flux.mean())
print(f"naive magnitude mean = {naive:.3f}")
print(f"intensity (flux) mean = {intensity:.3f}")
print(f"bias (naive - intensity) = {naive-intensity:+.3f} mag  (naive is too faint)")

naive magnitude mean = 10.890
intensity (flux) mean = 10.839
bias (naive - intensity) = +0.051 mag  (naive is too faint)


## 9. Lomb–Scargle power (period recovery)

**Plain language:** for each trial frequency, measure how well a sinusoid fits the
(unevenly sampled) data. The best-fitting frequency is the period. Watch for
aliases — always plot and look.

$$P(\omega)=\tfrac12\!\left[\frac{(\sum y_i\cos\omega(t_i-\tau))^2}
{\sum\cos^2\omega(t_i-\tau)}+
\frac{(\sum y_i\sin\omega(t_i-\tau))^2}{\sum\sin^2\omega(t_i-\tau)}\right]$$

In [10]:
from astropy.timeseries import LombScargle
# simulate CY Aqr: 3 hours of data, one 88-min period, sawtooth-ish
rng = np.random.default_rng(1)
t = np.sort(rng.uniform(0, 0.125, 200))          # ~3 hr = 0.125 d, ~2 cycles
phase = (t / P_CYAQR) % 1.0
y = 10.75 + 0.35*np.sign(0.3-phase)*np.abs(0.3-phase)**0.5 + rng.normal(0,0.02,len(t))
freq, power = LombScargle(t, y).autopower(minimum_frequency=1/0.15,
                                          maximum_frequency=1/0.02,
                                          samples_per_peak=10)
best = 1/freq[np.argmax(power)]
print(f"true period      = {P_CYAQR:.5f} d")
print(f"recovered period = {best:.5f} d  ({100*abs(best-P_CYAQR)/P_CYAQR:.1f}% off)")
print(f"(only ~2 cycles here -> imperfect; more cycles = better, per the notes)")

true period      = 0.06104 d
recovered period = 0.06440 d  (5.5% off)
(only ~2 cycles here -> imperfect; more cycles = better, per the notes)


## 10. Error propagation in quadrature

**Plain language:** independent errors combine as the square root of the sum of
squares, each weighted by how sensitively the result depends on that input.

$$\sigma_f^2 = \left(\frac{\partial f}{\partial x}\right)^2\sigma_x^2
+ \left(\frac{\partial f}{\partial y}\right)^2\sigma_y^2 + \cdots$$

In [11]:
# example: distance modulus mu = m - M - A_V, all partials = +/-1
m_err, M_err_v, A_err = 0.02, sig_tot, 0.05
sig_mu = np.sqrt(m_err**2 + M_err_v**2 + A_err**2)
print(f"sigma_m   = {m_err:.3f}")
print(f"sigma_M   = {M_err_v:.3f}  (dominant - the intrinsic scatter)")
print(f"sigma_A_V = {A_err:.3f}")
print(f"sigma_mu  = sqrt(sum of squares) = {sig_mu:.3f} mag")
print(f"\n-> the M_V error dominates, so better photometry barely helps")

sigma_m   = 0.020
sigma_M   = 0.253  (dominant - the intrinsic scatter)
sigma_A_V = 0.050
sigma_mu  = sqrt(sum of squares) = 0.259 mag

-> the M_V error dominates, so better photometry barely helps


## 11. Weighted linear least squares (deriving the relation)

**Plain language:** fit M_V = a + b·logP, weighting each star by
1/σ² so precise stars pull harder. Quality-discount lower-grade stars by
inflating their σ.

$$b=\frac{S\,S_{xy}-S_x S_y}{S\,S_{xx}-S_x^2}, \quad
a=\frac{S_{xx}S_y - S_x S_{xy}}{S\,S_{xx}-S_x^2}$$
where $S=\sum w$, $S_x=\sum w x$, etc., and $w=1/\sigma^2$.

In [12]:
# the project's 8-star sample (logP, M_V, quality)
stars = [
    ("CY Aqr", 0.06104, 2.71, "A"), ("ZZ Mic", 0.06718, 1.81, "B"),
    ("DY Peg", 0.07293, 2.24, "A"), ("GP And", 0.07868, 1.94, "B"),
    ("AE UMa", 0.08602, 1.76, "B"), ("YZ Boo", 0.10409, 1.67, "B"),
    ("XX Cyg", 0.13486, 1.45, "A"), ("DY Her", 0.14863, 1.21, "B"),
]
x = np.array([np.log10(s[1]) for s in stars])
y = np.array([s[2] for s in stars])
# quality-based sigma: A-grade tight, B-grade inflated
qmult = {"A":1.0, "B":1.5, "C":2.5}
sig = np.array([SIGMA_INTRINSIC * qmult[s[3]] for s in stars])
w = 1/sig**2

def wls(x, y, w):
    S=w.sum(); Sx=(w*x).sum(); Sy=(w*y).sum()
    Sxx=(w*x*x).sum(); Sxy=(w*x*y).sum()
    d=S*Sxx-Sx*Sx
    b=(S*Sxy-Sx*Sy)/d; a=(Sxx*Sy-Sx*Sxy)/d
    return a, b, np.sqrt(Sxx/d), np.sqrt(S/d)

a,b,ae,be = wls(x,y,w)
print(f"quality-weighted fit: M_V = {a:.3f}(+/-{ae:.3f}) + {b:.3f}(+/-{be:.3f})*logP")
print(f"Cohen & Sarajedini  : M_V = {A_CS}(+/-{A_ERR}) + {B_CS}(+/-{B_ERR})*logP")
# unweighted for contrast
au,bu,_,_ = wls(x,y,np.ones_like(w))
print(f"\nunweighted fit slope = {bu:.3f}  (population mixing flattens it)")
print(f"weighted fit slope   = {b:.3f}  (closer to published, A-grade dominates)")
print(f"the SPREAD is the honest systematic uncertainty")

quality-weighted fit: M_V = -1.399(+/-0.677) + -3.163(+/-0.636)*logP
Cohen & Sarajedini  : M_V = -1.64(+/-0.11) + -3.389(+/-0.09)*logP

unweighted fit slope = -2.879  (population mixing flattens it)
weighted fit slope   = -3.163  (closer to published, A-grade dominates)
the SPREAD is the honest systematic uncertainty


---
*This notebook mirrors the equations in the narrative reference PDF. Every value
is from the CY Aqr project so it runs offline. Change `P_CYAQR`, `V_MEAN`, etc. at
the top to explore other stars.*